# ⚙️ Notebook 2: Medical Imaging Preprocessing Pipeline

**Author:** Motaz Alqaoud, PhD | Senior AI/ML Engineer @ Abbott  
**GitHub:** [motazalqaoud](https://github.com/motazalqaoud)

---

## What you'll learn
- Intensity normalization methods and when to use each
- Resampling to isotropic voxel spacing
- Anatomy-aware data augmentation (why standard CV augmentation fails in medicine)
- Building a complete preprocessing pipeline for training

## Clinical context
> The #1 reason medical AI models fail in production is not the model architecture.  
> It's the preprocessing. A model trained on non-resampled data from Scanner A  
> will silently produce wrong predictions on Scanner B — with no error, just bad output.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import zoom, rotate
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..', 'src'))

# Reuse synthetic data from Notebook 1
def create_synthetic_mri(shape=(64, 128, 128), seed=42):
    np.random.seed(seed)
    volume = np.random.normal(0.2, 0.05, shape).astype(np.float32)
    z, y, x = np.mgrid[0:shape[0], 0:shape[1], 0:shape[2]]
    sphere = ((z-shape[0]//2)**2/(shape[0]//2.2)**2 +
              (y-shape[1]//2)**2/(shape[1]//2.2)**2 +
              (x-shape[2]//2)**2/(shape[2]//2.2)**2)
    volume[sphere < 1] += 0.4
    mask = np.zeros(shape, dtype=np.uint8)
    cz, cy, cx = shape[0]//2, shape[1]//3, shape[2]//3
    r = 12
    lesion = ((z-cz)**2/r**2 + (y-cy)**2/r**2 + (x-cx)**2/r**2) < 1
    volume[lesion] += 0.35
    mask[lesion] = 1
    volume += np.random.normal(0, 0.03, shape)
    return np.clip(volume, 0, 1).astype(np.float32), mask.astype(np.uint8)

volume, mask = create_synthetic_mri()
spacing = (5.0, 1.0, 1.0)  # Non-isotropic — 5mm axial slices
print(f"Volume: {volume.shape}, spacing: {spacing} mm")


## 1. Intensity Normalization — Choosing the Right Method

In [ ]:
def zscore_normalize(vol, mask=None):
    roi = vol[mask > 0] if mask is not None else vol.flatten()
    return (vol - roi.mean()) / (roi.std() + 1e-8)

def percentile_clip(vol, lo=1, hi=99):
    p_lo, p_hi = np.percentile(vol, lo), np.percentile(vol, hi)
    clipped = np.clip(vol, p_lo, p_hi)
    return (clipped - p_lo) / (p_hi - p_lo + 1e-8)

def minmax_normalize(vol):
    return (vol - vol.min()) / (vol.max() - vol.min() + 1e-8)

# Simulate a volume with outlier bright artifact
volume_w_artifact = volume.copy()
volume_w_artifact[10, 20, 20] = 5.0  # Bright artifact voxel

methods = {
    "Original": volume_w_artifact,
    "Min-Max": minmax_normalize(volume_w_artifact),
    "Percentile Clip (1-99%)": percentile_clip(volume_w_artifact),
    "Z-Score": zscore_normalize(volume_w_artifact),
}

cz = volume.shape[0] // 2
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for col, (name, vol) in enumerate(methods.items()):
    # Image
    axes[0, col].imshow(vol[cz], cmap='gray', vmin=vol[cz].min(), vmax=vol[cz].max())
    axes[0, col].set_title(name, fontsize=10, fontweight='bold')
    axes[0, col].axis('off')
    
    # Histogram
    axes[1, col].hist(vol.flatten(), bins=80, color='steelblue', alpha=0.8)
    axes[1, col].set_xlabel("Intensity", fontsize=9)
    if col == 0:
        axes[1, col].set_ylabel("Count", fontsize=9)

fig.suptitle("Normalization Methods Comparison (with bright artifact)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("📊 Which to use?")
print("  Min-Max:          Simple, BUT destroyed by artifacts/outliers")
print("  Percentile Clip:  Robust to outliers — recommended for MRI")
print("  Z-Score:          Best when using brain/organ mask for statistics")


## 2. Resampling to Isotropic Spacing

Our volume has 5mm axial spacing vs 1mm in-plane. This must be corrected before training.

In [ ]:
def resample_to_spacing(vol, current_spacing, target_spacing, order=1):
    """Resample volume to target voxel spacing."""
    zoom_factors = tuple(c / t for c, t in zip(current_spacing, target_spacing))
    resampled = zoom(vol, zoom_factors, order=order)
    return resampled.astype(np.float32), zoom_factors

target_spacing = (1.0, 1.0, 1.0)  # Isotropic 1mm
vol_resampled, factors = resample_to_spacing(volume, spacing, target_spacing)
mask_resampled, _ = resample_to_spacing(mask.astype(np.float32), spacing, target_spacing, order=0)
mask_resampled = (mask_resampled > 0.5).astype(np.uint8)

print(f"Original:   shape={volume.shape}, spacing={spacing}")
print(f"Resampled:  shape={vol_resampled.shape}, spacing={target_spacing}")
print(f"Zoom factors: {tuple(round(f,2) for f in factors)}")
print()

# Visualize the difference
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Sagittal view shows the anisotropy most clearly
for col, (name, vol_v, msk_v) in enumerate([
    ("Original (5mm axial)", volume, mask),
    ("After Resampling (1mm iso)", vol_resampled, mask_resampled),
]):
    # Sagittal
    cx = vol_v.shape[2] // 2
    axes[0, col].imshow(vol_v[:, :, cx].T, cmap='gray', aspect='auto')
    ov = np.zeros((*vol_v[:, :, cx].T.shape, 4))
    ov[msk_v[:, :, cx].T > 0] = [1, 0.2, 0.2, 0.5]
    axes[0, col].imshow(ov, aspect='auto')
    axes[0, col].set_title(f"{name}\nSagittal — shape: {vol_v.shape}", fontsize=9, fontweight='bold')
    axes[0, col].axis('off')
    
    # Axial
    cz = vol_v.shape[0] // 2
    axes[1, col].imshow(vol_v[cz], cmap='gray')
    ov2 = np.zeros((*msk_v[cz].shape, 4))
    ov2[msk_v[cz] > 0] = [1, 0.2, 0.2, 0.5]
    axes[1, col].imshow(ov2)
    axes[1, col].set_title(f"Axial", fontsize=9)
    axes[1, col].axis('off')

# Difference map
axes[0, 2].axis('off')
axes[0, 2].text(0.5, 0.5, 
    "Sagittal view reveals\nthe anisotropy:\n\nOriginal: squished vertically\n(5mm slices look thick)\n\nResampled: correct\nanatomical proportions",
    ha='center', va='center', fontsize=11, 
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1, 2].axis('off')

plt.suptitle("Effect of Resampling to Isotropic Spacing", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Anatomy-Aware Augmentation

Why standard CV augmentation is dangerous in medical imaging — and what to do instead.

In [ ]:
# ── Why random flipping is dangerous ──────────────────────────────────────

# Simulate a chest-like asymmetric volume (heart on left side)
asym_vol = vol_resampled.copy()
cy = asym_vol.shape[1] // 2
asym_vol[:, cy-20:cy+5, 15:40] += 0.3  # "Heart" on left side
asym_vol = np.clip(asym_vol, 0, 1)

# Bad augmentation: random horizontal flip
bad_flip = np.flip(asym_vol, axis=2).copy()

# Good augmentation: small rotation
good_aug = rotate(asym_vol, angle=8, axes=(1, 2), reshape=False, order=1)

cz = asym_vol.shape[0] // 2
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(asym_vol[cz], cmap='gray')
axes[0].set_title("Original\n(heart on left — correct)", fontweight='bold', fontsize=10)
axes[0].axis('off')

axes[1].imshow(bad_flip[cz], cmap='gray')
axes[1].set_title("❌ Random Horizontal Flip\n(heart on right — WRONG anatomy!)", 
                  fontweight='bold', fontsize=10, color='red')
axes[1].axis('off')

axes[2].imshow(good_aug[cz], cmap='gray')
axes[2].set_title("✅ Small Rotation (8°)\n(anatomy preserved)", 
                  fontweight='bold', fontsize=10, color='green')
axes[2].axis('off')

plt.suptitle("Why Random Flip is Dangerous in Medical Imaging", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Complete safe augmentation pipeline ───────────────────────────────────

def safe_augment(volume, mask, rotation_range=12, intensity_shift=0.08,
                 intensity_scale=0.08, noise_std=0.015, seed=None):
    """
    Anatomy-aware augmentation for medical images.
    Returns augmented volume and mask.
    """
    if seed is not None:
        np.random.seed(seed)
    
    aug_vol = volume.copy()
    aug_mask = mask.copy()
    
    # 1. Small in-plane rotation only
    angle = np.random.uniform(-rotation_range, rotation_range)
    aug_vol  = rotate(aug_vol,  angle, axes=(1, 2), reshape=False, order=1)
    aug_mask = rotate(aug_mask.astype(float), angle, axes=(1, 2), reshape=False, order=0)
    aug_mask = (aug_mask > 0.5).astype(np.uint8)
    
    # 2. Intensity augmentation (volume only)
    scale = np.random.uniform(1 - intensity_scale, 1 + intensity_scale)
    shift = np.random.uniform(-intensity_shift, intensity_shift)
    aug_vol = aug_vol * scale + shift
    
    # 3. Gaussian noise
    aug_vol += np.random.normal(0, noise_std, aug_vol.shape)
    
    return np.clip(aug_vol, 0, 1).astype(np.float32), aug_mask

# Show 6 augmented versions
cz = vol_resampled.shape[0] // 2
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes[0, 0].imshow(vol_resampled[cz], cmap='gray')
ov = np.zeros((*mask_resampled[cz].shape, 4))
ov[mask_resampled[cz] > 0] = [1, 0.2, 0.2, 0.5]
axes[0, 0].imshow(ov)
axes[0, 0].set_title("Original", fontweight='bold')
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for i in range(6):
    row, col = divmod(i, 3)
    aug_v, aug_m = safe_augment(vol_resampled, mask_resampled, seed=i*7)
    axes[row, col+1].imshow(aug_v[cz], cmap='gray')
    ov2 = np.zeros((*aug_m[cz].shape, 4))
    ov2[aug_m[cz] > 0] = [1, 0.2, 0.2, 0.5]
    axes[row, col+1].imshow(ov2)
    axes[row, col+1].set_title(f"Augmented #{i+1}", fontsize=9)
    axes[row, col+1].axis('off')

plt.suptitle("Safe Medical Augmentation — Anatomy Preserved", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Full Preprocessing Pipeline

Combining all steps into a single reusable function.

In [ ]:
def preprocess_volume(volume, spacing, mask=None,
                       target_spacing=(1.0, 1.0, 1.0),
                       norm_method='percentile',
                       augment=False):
    """
    Complete preprocessing pipeline for a medical imaging volume.
    
    Steps:
        1. Resample to target spacing
        2. Normalize intensity
        3. (Optional) Augment
    
    Args:
        volume:         3D numpy array
        spacing:        current voxel spacing (z, y, x) in mm
        mask:           optional segmentation mask
        target_spacing: desired isotropic spacing
        norm_method:    'percentile', 'zscore', or 'minmax'
        augment:        apply random augmentation
    
    Returns:
        preprocessed volume (and mask if provided)
    """
    # Step 1: Resample
    vol_r, _ = resample_to_spacing(volume, spacing, target_spacing, order=1)
    msk_r = None
    if mask is not None:
        msk_r, _ = resample_to_spacing(mask.astype(float), spacing, target_spacing, order=0)
        msk_r = (msk_r > 0.5).astype(np.uint8)
    
    # Step 2: Normalize
    if norm_method == 'percentile':
        vol_r = percentile_clip(vol_r)
    elif norm_method == 'zscore':
        vol_r = zscore_normalize(vol_r, msk_r)
    else:
        vol_r = minmax_normalize(vol_r)
    
    # Step 3: Augment
    if augment and msk_r is not None:
        vol_r, msk_r = safe_augment(vol_r, msk_r)
    elif augment:
        vol_r, _ = safe_augment(vol_r, np.zeros_like(vol_r, dtype=np.uint8))
    
    if mask is not None:
        return vol_r, msk_r
    return vol_r

# Run pipeline
vol_final, msk_final = preprocess_volume(
    volume, spacing, mask=mask,
    target_spacing=(1.0, 1.0, 1.0),
    norm_method='percentile',
    augment=False
)

print("Pipeline complete!")
print(f"  Input:  shape={volume.shape}, spacing={spacing}, range=[{volume.min():.2f}, {volume.max():.2f}]")
print(f"  Output: shape={vol_final.shape}, spacing=(1,1,1), range=[{vol_final.min():.2f}, {vol_final.max():.2f}]")
print(f"  Mask:   {msk_final.sum()} lesion voxels")


## ✅ Summary

| Step | Method | Key Insight |
|---|---|---|
| Normalization | Percentile clip | Robust to MRI artifacts |
| Resampling | scipy.zoom | Must match spacing before training |
| Augmentation | Small rotation + intensity | Never flip asymmetric anatomy |
| Full pipeline | `preprocess_volume()` | Plug in before your DataLoader |

**Next:** [Notebook 3 — Tumor Segmentation with U-Net](03_tumor_segmentation_unet.ipynb)
